In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch

print(torch.cuda.is_available())

True


In [ ]:
!pip install transformers accelerate sentencepiece -q

In [ ]:
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import torch

In [ ]:
df = pd.read_csv(
"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/manual_news_dataset.csv"
)

print(df.shape)

(80232, 42)


In [ ]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model.eval()

print("FinBERT Loaded")

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT Loaded
cuda


In [ ]:
labels = [
    "positive",
    "negative",
    "neutral"
]

def batch_finbert(texts, batch_size=64):

    positives = []
    negatives = []
    neutrals = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )

        inputs = {
            k:v.to(device)
            for k,v in inputs.items()
        }

        with torch.no_grad():

            outputs = model(**inputs)

            probs = torch.softmax(
                outputs.logits,
                dim=1
            )

        positives.extend(
            probs[:,0].cpu().numpy()
        )

        negatives.extend(
            probs[:,1].cpu().numpy()
        )

        neutrals.extend(
            probs[:,2].cpu().numpy()
        )

    return np.column_stack(
        (
            positives,
            negatives,
            neutrals
        )
    )

In [ ]:
results = batch_finbert(

    df["headline_1"]
    .fillna("NO_NEWS")
    .astype(str)
    .tolist()

)

df["h1_positive"] = results[:,0]

df["h1_negative"] = results[:,1]

df["h1_neutral"] = results[:,2]

In [ ]:
results = batch_finbert(

    df["headline_2"]
    .fillna("NO_NEWS")
    .astype(str)
    .tolist()

)

df["h2_positive"] = results[:,0]

df["h2_negative"] = results[:,1]

df["h2_neutral"] = results[:,2]

In [ ]:
results = batch_finbert(

    df["headline_3"]
    .fillna("NO_NEWS")
    .astype(str)
    .tolist()

)

df["h3_positive"] = results[:,0]

df["h3_negative"] = results[:,1]

df["h3_neutral"] = results[:,2]

In [ ]:
df["overall_positive"] = (

    df["h1_positive"]

    +

    df["h2_positive"]

    +

    df["h3_positive"]

)/3


df["overall_negative"] = (

    df["h1_negative"]

    +

    df["h2_negative"]

    +

    df["h3_negative"]

)/3


df["overall_neutral"] = (

    df["h1_neutral"]

    +

    df["h2_neutral"]

    +

    df["h3_neutral"]

)/3

In [ ]:
df["news_strength"] = (

    df["overall_positive"]

    -

    df["overall_negative"]

)

In [ ]:
def dominant(row):

    x = {
        "POSITIVE":row["overall_positive"],
        "NEGATIVE":row["overall_negative"],
        "NEUTRAL":row["overall_neutral"]
    }

    return max(
        x,
        key=x.get
    )

df["dominant_sentiment"] = (
    df.apply(
        dominant,
        axis=1
    )
)

In [ ]:
save_path = (
"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/manual_finbert_dataset.csv"
)

df.to_csv(
    save_path,
    index=False
)

print(df.shape)

print("FINBERT DATASET CREATED")

(80232, 56)
FINBERT DATASET CREATED
